### FastText
- Word2Vec 에서 OOV(사전에 없는 용어) 문제를 해결하기 위한 모델
- Word2Vec 에서는 '강아지' 와 '강아지들' 문구를 다른 단어로 취급
- FastText는 Word2Vec의 학습 방식과 비슷
    - 단어를 한글자씩 잘게 쪼개서 단어의 유사도를 생성
    - '강아지' -> '강', '아', '지', '강아', '아지', '강아지'
    (n-gram 방식)
    - word2vec 과 기본적인 매개변수는 같지만 min_n, max_n 매개변수가 존재
    - subword의 최소 길이와 최대 길이를 설정
    - min_n = 1, max_n = 1--> subword를 사용하지 않겠다 -> Word2Vec 같은 형태로 학습
    


In [1]:
from gensim.models import Word2Vec, FastText

In [4]:
# 샘플 문장 생성
sentences = [
    ['이커머스', '데이터', '분석', '진행'],
    ['상품', '리뷰', '감성','분석', '합니다'],
    ['형태소', '단위', '임베딩', '가능']
]

model = FastText(
    sentences=sentences,
    vector_size= 50,    # 단위벡터의 차원의 수
    window= 3,          # 주변의 확인할 단어의 개수
    min_count= 1,       # 최고 출현 횟수
    sg = 1,             # Skip-gram 방식으로 확률 계산
    epochs= 10,          # 학습 반복 횟수
    min_n= 2,
    max_n= 6
)

In [ ]:
# 특정 단어의 벡터를 확인
model.wv['이커머스']

In [6]:
# 단어 유사를 확인
model.wv.most_similar('데이터', topn=3)

[('이커머스', 0.267589807510376),
 ('리뷰', 0.14490611851215363),
 ('가능', 0.128218874335289)]

In [8]:
# 학습 내용에 없는 단어를 확인
model.wv['감정']

array([-5.5013164e-03, -2.2984277e-03, -2.5746368e-03,  4.0548560e-03,
        5.9943846e-03,  7.8274556e-05, -3.3544376e-04,  1.2387058e-03,
       -5.6768246e-03, -1.9010369e-03,  5.5367098e-04, -4.3595727e-03,
        1.8590713e-03, -1.6518653e-03,  1.2153205e-03,  4.9256259e-03,
       -1.0720404e-03,  1.6735172e-03, -7.5531616e-03,  9.4418990e-04,
       -3.0814782e-03, -4.9539101e-03,  6.7937217e-04,  2.4488673e-03,
        1.9780006e-03, -2.2291301e-03, -3.4820414e-04,  4.1451957e-04,
        7.0494846e-03,  1.9069883e-03, -4.7397362e-03,  1.0020768e-02,
        5.4202583e-03,  6.6833789e-03, -7.0033036e-03,  3.0828256e-03,
        4.4808798e-03,  1.0817961e-02,  4.6293014e-03,  2.7071584e-03,
       -4.2115343e-03, -4.7755619e-03,  2.4743190e-03, -2.6541452e-03,
        9.1218427e-03, -2.9160656e-04,  4.3158792e-04, -6.9790357e-03,
        4.6274574e-03,  2.2755212e-03], dtype=float32)

In [9]:
# Word2Vec 을 이용해서 sentences 학습하고 없는 단어 확인
model2 = Word2Vec(
    sentences=sentences,
    window=3,
    vector_size=50,
    min_count=1,
    sg=1,
    epochs=10
)

In [11]:
# 학습한 단어 출력
# model2.wv['감정']
# Word2Vec 은 없는 단를 단위벡터로 확인하면 에러 발생

- Word2Vec 과 FastText와 단어간의 유사도 차이 확인

    - '강아지', '강아지들' 두개의 단어를 유사한 단어
        - Word2Vec는 다른 단어로 인식 -> 유사도 낮게
        - FastText는 비슷한 단어로 인식 -> 유사도 높게

In [12]:
sentences2 = [
    ['고양이', '고양이들', '귀엽다','동물','반려동물'],
    ['강아지','강아지들','귀엽다','동물','반려동물'],
    ['달리다','달리는','달림','걷다','걷는'],
    ['빠르다','빠른','느리다','느림'],
    ['예쁘다','예쁨','예쁜','매력적이다'],
    ['컴퓨터','컴퓨팅','컴퓨터들','기계']
]

# 모델학습 (word2vec, fasttext)
w2v = Word2Vec(
    sentences=sentences2,
    vector_size=100,
    window=3,
    min_count=1,
    sg=1,
    epochs=50,
    seed=42
)

ft = FastText(
    sentences=sentences2,
    vector_size=100,
    window=3,
    min_count=1,
    sg=1,
    epochs=50,
    seed=42,
    min_n=2,
    max_n=6
)



In [13]:
print(
    'Word2Vec 유사도', w2v.wv.similarity('강아지', '강아지들')
)
# FastText
print(
    'FastText 유사도 : ', ft.wv.similarity('강아지', '강아지들')
)

Word2Vec 유사도 0.20342888
FastText 유사도 :  0.385201


In [15]:
# 비교 대상 단어들
text_text = [
    ['강아지', '강아지들'],
    ['고양이','고양이들'],
    ['달리다', '달리는'],
    ['예쁘다','예쁜'],
    ['컴퓨터', '컴퓨팅'],
    ['빠르다', '느림']
]

for a, b in text_text:
    w2v_sim = float(w2v.wv.similarity(a,b))
    ft_sim = float(ft.wv.similarity(a,b))
    print(f"{a} - {b}의 유사도 : {round(w2v_sim, 4)}, \
          Fasttext ({round(ft_sim, 4)})")

강아지 - 강아지들의 유사도 : 0.2034,           Fasttext (0.3852)
고양이 - 고양이들의 유사도 : -0.1262,           Fasttext (0.3965)
달리다 - 달리는의 유사도 : -0.0706,           Fasttext (0.2667)
예쁘다 - 예쁜의 유사도 : 0.1392,           Fasttext (0.0696)
컴퓨터 - 컴퓨팅의 유사도 : -0.0516,           Fasttext (0.3647)
빠르다 - 느림의 유사도 : 0.0108,           Fasttext (-0.0127)


In [17]:
# 상품 명을 기준으로 특정 상품을 검색 시 연관된 상품의 
# 목록을 확인한다
products = {
    'P001' : '무선 이어폰 블루투스 노이즈캔슬링 충전케이스',
    'P002' : '유선 이어폰 하이파이 금도금 플러그',
    'P003' : '게이밍 마우스 RGB 경량 디자인',
    'P004' : '무선 마우스 초경량 블루투스 듀얼모드',
    'P005' : '헤드폰 노이즈캔슬링 유선'
}

# products에서 FastText 통해 학습을 시키기 위해 데이터를 추출
# 필요한 데이터는 딕셔너리형 데이터에서 value 들이 필요
datas = products.values()
# data를 공백을 기준으로 나눠준다

tokens = []
for data in datas:
    tokens.append(data.split())

tokens

[['무선', '이어폰', '블루투스', '노이즈캔슬링', '충전케이스'],
 ['유선', '이어폰', '하이파이', '금도금', '플러그'],
 ['게이밍', '마우스', 'RGB', '경량', '디자인'],
 ['무선', '마우스', '초경량', '블루투스', '듀얼모드'],
 ['헤드폰', '노이즈캔슬링', '유선']]

In [19]:
ft2 = FastText(
    sentences= tokens,
    vector_size=100,
    window=3,
    min_count=1,
    sg=1,
    epochs= 10,
    min_n=3,
    max_n=6,
    seed=42
)

In [ ]:
# 단위 벡터 확인 -> vector_size
ft.wv['마우스']

In [22]:
import numpy as np
def sent_vec(token):
    vecs = []
    for w in token:
        vecs.append(ft.wv[w])
    # 해당 vecs가 존재하지 않는다면 희소행렬을 되돌려준다
    if not vecs:
        return np.zeros(ft.vector_size)
    
    v = np.mean(vecs, axis=0)
    # 일반적인 묹ㅇ의 평균 벡터를 구하는 식
    # 성능을 올리기 위해서 L2 정규화 -> 벡터의 거리로 나눠준다
    if type == 'l2':
        v= v/(np.linalg.norm(v)+1e-12)

    return v


In [35]:
# 토큰화 된 데이터들을 평균 벡터
item_vecs = []
for t in tokens:
    # print(sent_vec(t))
    # break
    item_vecs.append(sent_vec(t))

In [36]:
item_vecs

[array([ 5.68687101e-04, -4.47922677e-04,  2.25411568e-04, -9.38823563e-04,
        -1.35447620e-03,  5.69807598e-04, -1.06130261e-04, -1.56455988e-03,
        -4.49358340e-04, -1.93134096e-04,  4.12165275e-04,  6.66525040e-04,
        -3.92539136e-04,  3.18049773e-04, -1.24708749e-05,  1.11867755e-03,
         1.45315693e-03,  4.97972709e-04,  3.30269657e-04, -3.32839612e-04,
         2.00544364e-05, -1.38855452e-04,  5.39083441e-04,  6.64813328e-04,
         4.07810032e-04, -1.75284760e-04,  9.69403074e-04, -1.74880406e-04,
        -4.75434965e-04,  6.32041134e-04, -3.25016386e-04, -1.43916172e-04,
         2.55313615e-04, -1.06802047e-03, -3.52118514e-04,  1.58119627e-04,
         3.65634856e-04,  3.62791674e-04, -6.93874783e-04, -2.22496219e-05,
        -8.18753790e-04, -6.57265409e-05,  9.29408590e-04, -6.37178484e-04,
         6.49071590e-04,  1.41488941e-04,  9.09751281e-04,  6.66708394e-04,
        -9.57795244e-04, -8.23974493e-04,  5.92730066e-04,  7.08769294e-05,
        -9.0

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
idx = list(products.keys()).index('P003')
idx

2

In [38]:
# 코사인 유사도를 이용해서 가장 근접한 상품의
def recommend_by_text(product_id):
    # 해상 상품명의 위치 값 ->
    # item_vect 위치를 이용하여 코사인 유사도를 생성하기위함
    # 상품의 id값들은 products라는 dict에서 해당 id의 위치를 저장
    idx = list(products.keys()).index(product_id)

    # item_vecs에서 해당 인덱스의 값과 전체 vectors의 값을 비교
    # 코사인 유사도를 확인하면 사하느이 문장을 유사도를 확인했기 때문에 2차원이 아닌
    # 1차원 데이터를 생성
    sims = cosine_similarity(item_vecs[idx:idx+1], item_vecs).ravel()
    # 내림차순 정렬
    order = sims.argsort()[::-1]
    # 추천 단어들을 출력
    rec = []
    for i in order:
        if list(products.keys())[i] != product_id:
            # 상품의 ID와 유사도의 값들을 rec에 추가
            rec.append([list(products.keys())[i], sims[i]])
    return rec

In [40]:
rec_list = recommend_by_text('P001')

In [53]:
for idx, (pid, _) in enumerate(rec_list):
    # 가장 유사도가 높은 상위 2개 상품의 이름을 확인
    print(products[pid])
    if idx ==1:
        break

무선 마우스 초경량 블루투스 듀얼모드
유선 이어폰 하이파이 금도금 플러그


In [54]:
# 세션에 따른 물품 추천
sessions = [
    ['P001', 'P004'],
    ['P001', 'P002'],
    ['POO3', 'P004'],
    ['P005', 'P001'],
    ['P002', 'P005'],
    ['P003', 'P004', 'P001']
]

In [64]:
    # 식별자인 id를 기준으로 임베딩을 하는 경우에는 subword 불필요
ft_item = FastText(
    sentences= sessions,
    sg = 1,
    vector_size= 100,window=3,
    min_count=1,
    epochs=50,
    min_n=1,
    max_n=1,
    seed=42
)

w2v_item = Word2Vec(
    sentences= sessions,
    sg = 1,
    vector_size= 100,
    window=3,
    min_count=1,
    epochs=50,
    seed=42
)

In [65]:
def recommend_by_session(product_id, text_model, n = 3):
    # product_id : 상품의 id (첫번째 검색하는 아이템의 id)
    # text_model : 학습이 된 모델
    # n : 유사도 높은 상위의 n개를 출력
    # 유사한 단어를 출력해주는 함수 (most_similar)
    recs = text_model.wv.most_similar( product_id, topn = n )
    # most_similar() 의 결과값은 [(item_id, 유사도), ...]
    return recs

In [66]:
recommend_by_session('P001', ft_item)

[('P004', 0.8054043650627136),
 ('P005', 0.7459513545036316),
 ('P003', 0.7164985537528992)]

In [67]:
recommend_by_session('P002', w2v_item)

[('P005', 0.10437855124473572),
 ('P003', 0.08919382840394974),
 ('P001', -0.008333135396242142)]